In [1]:
# Imports, Setup and Parameters
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
import sys
import os
from tqdm.auto import tqdm  

# Jupyter magic command to display plots inline
%matplotlib inline

# Registers flagged for removal due to frequent sentence splitting errors 
# that could undermine result reliability.
REGISTERS_TO_IGNORE = [
    "Proposal of Binding Precedent",
    "Appellate Decision Records",
    "Juridical Topic Publication",
    "Study of Precedents",
    "Juridical Speech"
]

# Random seed for reproducibility
REPRODUCIBILITY_SEED = 1569832526



In [2]:
# Download carol-domain-sents

print("Downloading dataset 'carolina-c4ai/carol-domain-sents' from Hugging Face...")
try:
    dataset_dict = load_dataset("carolina-c4ai/carol-domain-sents")
    print(f"Successfully loaded dataset. Found splits: {list(dataset_dict.keys())}")
except Exception as e:
    print(f"Error downloading or loading dataset: {e}", file=sys.stderr)

Successfully loaded dataset. Found splits: ['train', 'test', 'hps']


In [3]:
# Combine Splits and Convert to DataFrame

print("Converting and combining all dataset splits into a Pandas DataFrame...")

all_dfs = []
for split_name, split_dataset in dataset_dict.items():
    print(f"Processing split: '{split_name}'...")
    all_dfs.append(split_dataset.to_pandas())

# Concatenate all DataFrames from all splits
df = pd.concat(all_dfs, ignore_index=True)

print(f"Successfully combined all splits. Total entries: {len(df)}")

# Check for required columns
required_columns = ['source_typology', 'domain']
if not all(col in df.columns for col in required_columns):
    print(f"Error: The dataset does not contain the required columns ({required_columns}).", file=sys.stderr)
else:
    # Display the first few rows and info of the combined DataFrame
    print("\nDataFrame Info:")
    df.info()
    print("\nDataFrame Head:")
    display(df.head())

Converting and combining all dataset splits into a Pandas DataFrame...
Processing split: 'train'...
Processing split: 'test'...
Processing split: 'hps'...
Successfully combined all splits. Total entries: 4531553

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4531553 entries, 0 to 4531552
Data columns (total 4 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   source_typology    object
 1   carolina_typology  object
 2   domain             object
 3   text               object
dtypes: object(4)
memory usage: 138.3+ MB

DataFrame Head:


,source_typology,carolina_typology,domain,text
0,#NEWS_JOU_W,#DATASETS_AND_OTHER_CORPORA,Journalistic,"Para prosperar, a candidatura de Hamburgo prec..."
1,#DISCUSSION_VIR_W,#WIKIS,Virtual,Acreditando tratar-se da força de Antônio Nett...
2,#VOCABULARY_ENTRY_INS_W,#WIKIS,Instructional,"Iniciou-se como escritora ao publicar ""13 Cont..."
3,#USER_PAGE_VIR_W,#WIKIS,Virtual,"Chegando a Santa Maria, podemos visitar o camp..."
4,#USER_PAGE_VIR_W,#WIKIS,Virtual,"Além disto, e considerando o alto grau de frag..."


In [4]:
# Calculate the count for each 'source_typology'
counts_per_typology = df.groupby('source_typology')['source_typology'].transform('count')

# Filter the original dataframe
# Keep only the rows where the group's count is 3000 or more.
df_filtered = df[counts_per_typology >= 3000]

# Check the result
print(f"Original DataFrame had {len(df)} entries.")
print(f"Filtered DataFrame has {len(df_filtered)} entries.")

# Verify which typologies remain:
print("\nRemaining typologies in the filtered dataframe:")
print(df_filtered['source_typology'].value_counts())

print("\nFiltered DataFrame Head:")
display(df_filtered.head())

Original DataFrame had 4531553 entries.
Filtered DataFrame has 4521462 entries.

Remaining typologies in the filtered dataframe:
source_typology
#SUBTITLE_ENT_S                                           906191
#VOCABULARY_ENTRY_INS_W                                   879179
#NEWS_JOU_W                                               838360
#APPELLATE_DECISION_RECORDS_JUR_W                         634366
#USER_PAGE_VIR_W                                          347382
#TWEET_VIR_W                                              299602
#DISCUSSION_VIR_W                                         254180
#REQUEST_FOR_PROPOSALS_JUR_W                              102446
#SCIENTIFIC_NEWS_JOU_W                                     56676
#STUDY_OF_PRECEDENTS_BY_MINISTER_JUD_W                     51292
#OPEN_COURT_HEARING_JUR_W                                  38796
#TOPICAL_PUBLICATION_JUR_W                                 29894
#REPORT_JUR_W                                              24696
#EDUCATION

,source_typology,carolina_typology,domain,text
0,#NEWS_JOU_W,#DATASETS_AND_OTHER_CORPORA,Journalistic,"Para prosperar, a candidatura de Hamburgo prec..."
1,#DISCUSSION_VIR_W,#WIKIS,Virtual,Acreditando tratar-se da força de Antônio Nett...
2,#VOCABULARY_ENTRY_INS_W,#WIKIS,Instructional,"Iniciou-se como escritora ao publicar ""13 Cont..."
3,#USER_PAGE_VIR_W,#WIKIS,Virtual,"Chegando a Santa Maria, podemos visitar o camp..."
4,#USER_PAGE_VIR_W,#WIKIS,Virtual,"Além disto, e considerando o alto grau de frag..."


In [5]:
# Dataframe Adaptation
# Rename the 'source_typology' column to 'register'
df_processed = df_filtered.rename(columns={'source_typology': 'register'})

# Define the mapping structure for the 'register' values
register_map = {
    "#SUBTITLE_ENT_S": "Subtitle",
    "#VOCABULARY_ENTRY_INS_W": "Vocabulary Entry",
    "#APPELLATE_DECISION_RECORDS_JUR_W": "Appellate Decision Records",
    "#USER_PAGE_VIR_W": "User Page",
    "#TWEET_VIR_W": "Tweet",
    "#DISCUSSION_VIR_W": "Virtual Discussion",
    "#REQUEST_FOR_PROPOSALS_JUR_W": "Request for Proposals",
    "#SCIENTIFIC_NEWS_JOU_W": "Scientific News",
    "#STUDY_OF_PRECEDENTS_BY_MINISTER_JUD_W": "Study of Precedents",
    "#OPEN_COURT_HEARING_JUR_W": "Open Court Hearing",
    "#TOPICAL_PUBLICATION_JUR_W": "Juridical Topic Publication",
    "#REPORT_JUR_W": "Juridical Report",
    "#EDUCATIONAL_RESOURCES_INS_W": "Educational Resources",
    "#ARTICLE_JOU_W": "Article",
    "#SPEECH_JUD_S": "Juridical Speech",
    "#TRAVEL_GUIDE_INS_W": "Travel Guide",
    "#ACTIVITIES_ORGANIZATION_AND_EXPERIENCES_SHARING_VIR_W": "Virtual Activities Organization and Experiences Sharing",
    "#PRECEDENTS_BULLETIN_JUD_W": "Precedents Bulletin",
    "#PROPOSAL_OF_BINDING_PRECEDENT_JUD_W": "Proposal of Binding Precedent",
    "#NEWS_JOU_W": "News"
}

# Apply the transformation to the 'register' column
df_processed['register'] = df_processed['register'].replace(register_map)

# Remove specific registers based on predefined list REGISTERS_TO_IGNORE:
# Calculate initial shape for verification
initial_rows = df_processed.shape[0]

# Filter the dataframe: Keep rows where 'register' is NOT in the ignore list
df_processed = df_processed[~df_processed['register'].isin(REGISTERS_TO_IGNORE)]

# Calculate final shape
final_rows = df_processed.shape[0]

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------

print("Verification of column rename:")
print(f"Original columns: {df_filtered.columns.tolist()}")
print(f"New columns: {df_processed.columns.tolist()}")

print(f"\nFiltering Report:")
print(f"Rows before filtering: {initial_rows}")
print(f"Rows after removing {REGISTERS_TO_IGNORE}: {final_rows}")
print(f"Rows removed: {initial_rows - final_rows}")

print("\nVerification of new 'register' values:")
print(df_processed['register'].value_counts().head(20))

print("\nProcessed DataFrame Head:")
display(df_processed.head())

Verification of column rename:
Original columns: ['source_typology', 'carolina_typology', 'domain', 'text']
New columns: ['register', 'carolina_typology', 'domain', 'text']

Filtering Report:
Rows before filtering: 4521462
Rows after removing ['Proposal of Binding Precedent', 'Appellate Decision Records', 'Juridical Topic Publication', 'Study of Precedents', 'Juridical Speech']: 3794672
Rows removed: 726790

Verification of new 'register' values:
register
Subtitle                                                   906191
Vocabulary Entry                                           879179
News                                                       838360
User Page                                                  347382
Tweet                                                      299602
Virtual Discussion                                         254180
Request for Proposals                                      102446
Scientific News                                             56676
Open Court H

,register,carolina_typology,domain,text
0,News,#DATASETS_AND_OTHER_CORPORA,Journalistic,"Para prosperar, a candidatura de Hamburgo prec..."
1,Virtual Discussion,#WIKIS,Virtual,Acreditando tratar-se da força de Antônio Nett...
2,Vocabulary Entry,#WIKIS,Instructional,"Iniciou-se como escritora ao publicar ""13 Cont..."
3,User Page,#WIKIS,Virtual,"Chegando a Santa Maria, podemos visitar o camp..."
4,User Page,#WIKIS,Virtual,"Além disto, e considerando o alto grau de frag..."


In [6]:
# Balancing

print(f"Original DataFrame size: {len(df_processed)}")

# Group by the 'register' column and sample 3000 entries
try:
    df_sampled = df_processed.groupby('register').sample(
        n=3000,
        random_state=REPRODUCIBILITY_SEED
    )
except ValueError as e:
    print(f"Error during sampling: {e}")
    df_sampled = None 

if df_sampled is not None:
    # Reset the index
    df_sampled = df_sampled.reset_index(drop=False)

    # Verify the results
    print(f"Final sampled DataFrame size: {len(df_sampled)}")

    print("\nCounts per 'register' in the new sampled DataFrame (should all be 3000):")
    print(df_sampled['register'].value_counts())

    print("\nSampled DataFrame Head:")
    display(df_sampled.head())

Original DataFrame size: 3794672
Final sampled DataFrame size: 45000

Counts per 'register' in the new sampled DataFrame (should all be 3000):
register
Article                                                    3000
Educational Resources                                      3000
Juridical Report                                           3000
News                                                       3000
Open Court Hearing                                         3000
Precedents Bulletin                                        3000
Request for Proposals                                      3000
Scientific News                                            3000
Subtitle                                                   3000
Travel Guide                                               3000
Tweet                                                      3000
User Page                                                  3000
Virtual Activities Organization and Experiences Sharing    3000
Virtual Discussi

,index,register,carolina_typology,domain,text
0,137935,Article,#UNIVERSITY_DOMAINS,Journalistic,"Ou seja, que não se deve culpar a ninguém e nã..."
1,3165238,Article,#UNIVERSITY_DOMAINS,Journalistic,"Não tinha intuito de os desarmar, nem de trazê..."
2,291534,Article,#UNIVERSITY_DOMAINS,Journalistic,Álvaro Pina; Ivana Jinkings.
3,1556009,Article,#UNIVERSITY_DOMAINS,Journalistic,"Em vários países, o ensino superior privado re..."
4,4191565,Article,#UNIVERSITY_DOMAINS,Journalistic,Quando ouve do jovem agente comunitário que a ...


In [7]:
# Compute the coverage of each register in the sample

counts_before_sampling = df_processed['register'].value_counts()
counts_after_sampling = df_sampled['register'].value_counts()

# Create a new DataFrame for the coverage analysis
coverage_df = pd.DataFrame({
    'Total_Entries': counts_before_sampling,
    'Sampled_Entries': counts_after_sampling
})

# Calculate the 'Coverage' ratio
coverage_df['Coverage_Ratio'] = coverage_df['Sampled_Entries'] / coverage_df['Total_Entries']
# Add a formatted percentage column
coverage_df['Coverage_Percentage'] = coverage_df['Coverage_Ratio'].apply(lambda x: f"{x:.2%}")
#  Sort the DataFrame by index (register name)
coverage_df = coverage_df.sort_index()
# Display
print("\n--- Coverage Analysis DataFrame (Full) ---")
display(coverage_df)

# Select and prepare the DataFrame columns for the LaTeX export
latex_output_df = coverage_df[['Total_Entries', 'Coverage_Percentage']].copy()

# Rename columns and index
latex_output_df.columns = ['Total Entries', 'Coverage (%)']
latex_output_df.index.name = 'Register' # Set the index name

# Generate the LaTeX string
# The 'column_format' 'lrr' means:
# l = left-aligned (for the Register names)
# r = right-aligned (for the two numeric/percentage columns)
latex_table_string = latex_output_df.to_latex(
    caption='Data Coverage per Register After Sampling',
    label='tab:coverage_analysis',
    column_format='lrr',
    bold_rows=True
)

# Export coverage table
output_dir = "tables"
os.makedirs(output_dir, exist_ok=True)
output_filename = os.path.join(output_dir,'coverage_table.tex')
try:
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write(latex_table_string)
    print(f"\n--- LaTeX Table Saved ---")
    print(f"Successfully saved the LaTeX table to '{output_filename}'")
except IOError as e:
    print(f"\n--- Error ---")
    print(f"Failed to write LaTeX table to file: {e}")


--- Coverage Analysis DataFrame (Full) ---


,Total_Entries,Sampled_Entries,Coverage_Ratio,Coverage_Percentage
register,,,,
Article,11127,3000,0.269614,26.96%
Educational Resources,19295,3000,0.155481,15.55%
Juridical Report,24696,3000,0.121477,12.15%
News,838360,3000,0.003578,0.36%
Open Court Hearing,38796,3000,0.077328,7.73%
Precedents Bulletin,4614,3000,0.650195,65.02%
Request for Proposals,102446,3000,0.029284,2.93%
Scientific News,56676,3000,0.052932,5.29%
Subtitle,906191,3000,0.003311,0.33%



--- LaTeX Table Saved ---
Successfully saved the LaTeX table to 'tables/coverage_table.tex'


In [8]:
# Saving the dataset to .csv
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)
df_sampled.to_csv(os.path.join(output_dir, "working_sample.csv"), index=False)